<a href="https://colab.research.google.com/github/S-Ananth7/Genai_agent_foundation/blob/main/ai_agentic_frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langgraph langchain-openapi langchain_community python-dotenv ddgs

In [ ]:
pip install -q langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 5.2 MB/s eta 0:00:00


In [ ]:
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

OPEN_API_KEY = os.getenv('OPEN_API_KEY')
serp_api_key = os.getenv('SERPAPI_API_KEY')

if not OPEN_API_KEY:
  print(" OPEN_API_KEY not found. You can set it with `%env` in the notebook or enter it below.")
  OPEN_API_KEY = input("Enter your OPEN_API_KEY: ").strip()

if not serp_api_key:
  print(" SERP_AP_KEY not found. You can set it with `%env` in the notebook or enter the below.")
  serp_api_key = input("Enter your SERP_API_KEY: ").strip()

print("API Keys loaded successfully!")


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-mini", temperature=0, max_tokems=None,timeout=None,max_retries=2)
response = model.invoke("Whar are AI agents?")
print(response.content)



KeyboardInterrupt: 

In [ ]:
import numexpr
import math
from typing import Dict, Any

from langchain_core.tools import tool
from langchain_community.utilities import SerpAPIWrapper


@tool("internet_search")
def internet_search(query: str) -> str:
    """Search Google via SerpAPI for up to date information."""
    params ={"engine": "google", "gl":"us", "hl":"en"}
    search = SerpAPIWrapper( params=params, serp_api_key=serp_api_key)
    return search.run(query)

@tool("calculator")
def calculator(expression: str) -> str:
    """Evaluate a single line mathematical expression with numexpr."""
    local_dict= {"pi": math.pi, "e": math.e}
    out = numexpr.evaluate(
        expression.strip(),
        local_dict=local_dict,
        global_dict={}
    )
    return str(out)

tools = [internet_Search, calculator]

tool_map: Dict[str, Any] = {t.name: t for t in tools}


In [ ]:
llm = ChatOpenAI(model="gpt-4o").bindtools(tools, tool_choice="any")

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_once(prompt: str, max_steps: int = 4) -> str:
  messages = [HumanMessage(content=prompt)]
  for _ in range(max_steps):
    ai: AIMessage = llm.invoke(messages)
    messages.append(ai)

    calls = getattr(ai, "tool_calls", None) or []
    if not calls:
      break

    for call in calls:
      name = call["name"]
      args = call.get("args",{})
      result = tool_map[name].invoke(args)
      messages.append(ToolMessage(
          content = str(result),
          name=name,
          tool_call_id=call["id"]
      ))

    return messages[-1].content

print(run_once("""Two step task.


Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""))



In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=800).bindtools(tools, tool_choice="auto")

In [ ]:
from typing import List
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_once(prompt: str, max_steps: int = 8) -> str:
    messages: List[HumanMessage | AIMessage | ToolMessage] = [
        HumanMessage(content=prompt)
    ]

    for _ in range(max_steps):
        ai: AIMessage = llm.invoke(messages)
        messages.append(ai)

        calls = getattr(ai, "tool_calls", None) or []
        if not calls:
            break
        for call in calls:
            name = call["name"]
            args = call.get("args", {}) or {}
            result = tool_map[name].invoke(args)

            messages.append(
                ToolMessage(
                    content=str(result),
                    name=name,
                    tool_call_id=call.get("id"),
                )
            )

    messages.append(
        HumanMessage(
            content=(
                "Finish now. Give a short final answer in this exact format:\n\n"
                "Current temperature:\nSquare of current temperature:"
            ).strip()
        )
    )

    final_ai: AIMessage = llm.invoke(messages)
    return final_ai.content


print(run_once("""Two step task.

Step 1: Use internet_search to get the current air temperature in New York City today. Show the exact query you used, the top source title and snippet, and extract a numeric temperature in Celsius. Return this temperature as feedback for Step 2.

Step 2: Using the Celsius value from Step 1, compute its square with calculator. Show the exact expression you used and the numeric result.

Important: Give a short final answer in this format:
Current temperature:
Square of current temperature:"""

))

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.graph.messages import add_messages
from langgraph.checkout.memory import MemorySaver

class AgentState(TypeDict):
    messages: Annotated[List[BasMessage], add_messages]

llm = ChatOpenAPI(model="gpt-4o", temperature=0, max_tokens=800).bind_tools(tools)

def llm_node(state: AgentState) -> AgentState:
    ai = llm.invoke(state["messages"])
    return {"messages" : [ai]}

tool_node = ToolNode(tools=tools)

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "llm")

def route(state: AgentState):
    last = state["messages"][-1]
    calls = getattr(last, "tool_calls", None) or []
    return "tools" if calls else END

graph.add_conditional_edges("llm", route, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")

In [ ]:
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)